# Optimize Weld's Experiment with Ax


In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig

In [ ]:
# Bootstrap MSVC toolchain inside the kernel process so BoTorch can build fused kernels on Windows.


def _load_msvc_env_from_vcvars() -> bool:
    vcvars = Path(
        r"C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\VC\Auxiliary\Build\vcvars64.bat"
    )
    if not vcvars.exists():
        print(f"MSVC bootstrap skipped: {vcvars} not found.")
        return False

    dump_env_cmd = Path(tempfile.gettempdir()) / "_dump_msvc_env.cmd"
    dump_env_cmd.write_text(
        f'@echo off\ncall "{vcvars}" >nul\nif errorlevel 1 exit /b 1\nset\n',
        encoding="utf-8",
    )

    try:
        proc = subprocess.run(
            ["cmd.exe", "/d", "/c", str(dump_env_cmd)],
            capture_output=True,
            text=True,
            encoding="mbcs",
            errors="replace",
            check=True,
        )
    except subprocess.CalledProcessError as exc:
        print(f"MSVC bootstrap failed with return code {exc.returncode}.")
        if exc.stderr:
            print(exc.stderr.strip())
        return False

    for line in proc.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value

    # Ensure the kernel keeps the active venv Scripts folder on PATH.
    scripts_dir = str(Path(sys.executable).resolve().parent)
    path_parts = os.environ.get("PATH", "").split(os.pathsep)
    if scripts_dir not in path_parts:
        os.environ["PATH"] = scripts_dir + os.pathsep + os.environ.get("PATH", "")

    return True


loaded = _load_msvc_env_from_vcvars()
print("MSVC env loaded:", loaded)
print("cl:", shutil.which("cl"))
print("ninja:", shutil.which("ninja"))

In [ ]:
# 1. Initialize the Client.
client = Client(random_seed=42)

In [ ]:
folder_path = Path(os.getcwd()) / "data"

In [ ]:
exp = pd.read_excel(folder_path / "experiments.xlsx")
display(exp)


In [ ]:
print(exp.columns)
cols_advancing = [
    "sigma_advancing_1",
    "sigma_advancing_2",
    "sigma_advancing_3",
]
cols_retreating = [
    "sigma_retreating_1",
    "sigma_retreating_2",
    "sigma_retreating_3",
]

In [ ]:
filt_advancing = exp[cols_advancing].isna().all(axis=1)
filt_retreating = exp[cols_retreating].isna().all(axis=1)
filt_removed = filt_advancing | filt_retreating
filt_removed

In [ ]:
exp_todo = exp[filt_removed].copy()
exp = exp[~filt_removed].copy()
display(exp)
display(exp_todo)

In [ ]:
# 2. Configure where Ax will search.

bound_R = [1400, 2000]  # Rotatinal speed
bound_W = [40, 100]  # Welding speed
bound_D = [10, 30]  # Dwell time

parameters = [
    RangeParameterConfig(
        name="R1",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="R2",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="R3",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W1",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W2",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W3",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="D",
        bounds=bound_D,
        parameter_type="int",
    ),
]

client.configure_experiment(
    name="parn_exp",
    parameters=parameters,
)

In [ ]:
MAX_SIGMA_ADVANCING = 250
MAX_SIGMA_RETREATING = 250

client.configure_optimization(
    objective="-sigma_advancing, -sigma_retreating",  # TODO: double check the sign
    outcome_constraints=[
        f"sigma_advancing <= {MAX_SIGMA_ADVANCING}",
        f"sigma_retreating <= {MAX_SIGMA_RETREATING}",
        "visual_quality >= 0.5",
    ],
)

In [ ]:
def cal_mean_sem(row, cols):
    filt = row[cols].notna()
    arr = row[cols][filt].astype(float).values

    if len(arr) <= 1:
        raise ValueError("Not enough data points to calculate mean and SEM.")

    mean = arr.mean()
    # NOTE: The ddof=1 because this is a sample standard deviation, not population.
    sem = arr.std(ddof=1) / np.sqrt(len(arr))
    return mean, sem

In [ ]:
for idx, row in exp.iterrows():
    trial_index = client.attach_trial(
        parameters={
            "R1": int(row["R1"]),
            "R2": int(row["R2"]),
            "R3": int(row["R3"]),
            "W1": int(row["W1"]),
            "W2": int(row["W2"]),
            "W3": int(row["W3"]),
            "D": int(row["D"]),
        }
    )
    advancing_mean, advancing_sem = cal_mean_sem(row, cols_advancing)
    retreating_mean, retreating_sem = cal_mean_sem(row, cols_retreating)
    visual_quality = float(row["visual_quality"])
    print(f"Trial {trial_index}: mean={advancing_mean}, sem={advancing_sem}")
    print(f"Trial {trial_index}: mean={retreating_mean}, sem={retreating_sem}")
    print(f"Trial {trial_index}: visual_quality={visual_quality}")
    client.complete_trial(
        trial_index=trial_index,
        raw_data={
            "sigma_advancing": (advancing_mean, advancing_sem),
            "sigma_retreating": (retreating_mean, retreating_sem),
            "visual_quality": visual_quality,
        },
    )

In [ ]:
data = client.get_next_trials(max_trials=10)
print("Current node:", client._generation_strategy.current_node.name)
cards = client.compute_analyses(display=True)

In [ ]:
# print(f"Rows attached: {len(exp)}")
# print(f"Rows filtered to todo: {len(exp_todo)}")

In [ ]:
# for node in client._generation_strategy._nodes:
#     if node.name == "Sobol":
#         for tc in node.transition_criteria:
#             print(tc.threshold, tc.transition_to)